> **Traçabilité (fusion Codes_finaux) :** notebook présent uniquement dans `Colab_Notebooks/spam_4_DNN_Keras_tuner.ipynb` (renommé ici `spam_4_DNN_Keras_tuner_Colab_enrichi.ipynb` pour éviter toute confusion avec `spam_4_DNN_Keras_tuner.ipynb`, fusion de la version prof). Variante Colab enrichie : installe `keras-tuner` via pip, config d'hyperparamètres KerasTuner bien plus détaillée et commentée (`n_layers_min/max`, `n_units_min/max/step`, `dropout_min/max/step`, `patience`, `EarlyStopping`). Contient une cellule en erreur (exécution incomplète, sortie stderr conservée).

---
# DNN : `spam` (avec `Keras` et `KerasTuner`)
---

## Packages

In [4]:
%pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.2 MB/s eta 0:00:00


In [5]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from keras import callbacks, layers, models

from keras_tuner.tuners import RandomSearch

---
## 1. Données
---

### 1.1. Importation

In [9]:
# En local :
# directory = '/Users/vincentlefieux/Dropbox/Docs_ACADEMIQUE/Data/'

# Sur Google collab ou Onyxia (sur un répertoire temporaire) :
# directory = ''

# Sur Google collab (sur le drive) :
from google.colab import drive
drive.mount('/content/drive')
directory = '/content/drive/MyDrive/Colab_Notebooks/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
data = pd.read_csv(directory + 'spam.csv',
                   header    = 0,
                   index_col = None,
                   sep       = ',',
                   decimal   = '.')

In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4601 entries, 0 to 4600
Data columns (total 58 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   word_freq_make              4601 non-null   float64
 1   word_freq_address           4601 non-null   float64
 2   word_freq_all               4601 non-null   float64
 3   word_freq_3d                4601 non-null   float64
 4   word_freq_our               4601 non-null   float64
 5   word_freq_over              4601 non-null   float64
 6   word_freq_remove            4601 non-null   float64
 7   word_freq_internet          4601 non-null   float64
 8   word_freq_order             4601 non-null   float64
 9   word_freq_mail              4601 non-null   float64
 10  word_freq_receive           4601 non-null   float64
 11  word_freq_will              4601 non-null   float64
 12  word_freq_people            4601 non-null   float64
 13  word_freq_report            4601 

In [12]:
data.head()

,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,...,char_freq_semicolon,char_freq_leftbrac,char_freq_leftsquarebrac,char_freq_exclaim,char_freq_dollar,char_freq_pound,capital_run_length_average,capital_run_length_longest,capital_run_length_total,spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191,1


### 1.2. Gestion des données manquantes

In [13]:
missing_percentage = data.isna().mean() * 100

print('MISSING VALUES :')
if missing_percentage[missing_percentage != 0].empty:
    print('No')
else:
    print(missing_percentage[missing_percentage != 0].sort_values(ascending=False))

MISSING VALUES :
No


### 1.3. Gestion des variables

In [14]:
target = 'spam'

y = data[target]
X = data.drop(target, axis=1)

### 1.4. Création des échantillons de validation et test

In [15]:
test_portion  = 1/5
valid_portion = 1/5

X_train_valid, X_test, y_train_valid, y_test = train_test_split(X, y, test_size=test_portion, random_state=1234)

X_train, X_valid, y_train, y_valid = train_test_split(X_train_valid, y_train_valid, test_size=valid_portion)

print('Dimensions de X_train :', X_train.shape)
print('Dimensions de X_valid :', X_valid.shape)
print('Dimensions de X_test  :', X_test.shape)

print('Dimensions de y_train :', y_train.shape)
print('Dimensions de y_valid :', y_valid.shape)
print('Dimensions de y_test  :', y_test.shape)

Dimensions de X_train : (2944, 57)
Dimensions de X_valid : (736, 57)
Dimensions de X_test  : (921, 57)
Dimensions de y_train : (2944,)
Dimensions de y_valid : (736,)
Dimensions de y_test  : (921,)


### 1.5. Normalisation des covariables

On normalise (centrage-réduction) les covariables :

In [16]:
scaler = StandardScaler(with_mean=True, with_std=True)
scaler.fit(X_train)

X_train_norm = scaler.transform(X_train)
X_valid_norm = scaler.transform(X_valid)
X_test_norm  = scaler.transform(X_test)

---
## 2. DNN
---

### 2.1. Architecture et optimiseur

Pour optimisation du réseau, on définit différents paramétrages sur les hyperparamètres:
- nb de couches
- nb de neurones
- proba pour drop out

In [17]:
dim_outputs = 1           # Nombre de classes de sortie (1 pour la classification binaire 'spam' ou 'non-spam')

n_layers_min = 1          # Nombre minimal de couches cachées pour KerasTuner
n_layers_max = 3          # Nombre maximal de couches cachées pour KerasTuner

n_units_min  = 32         # Nombre minimal de neurones par couche cachée
n_units_max  = 128        # Nombre maximal de neurones par couche cachée
n_units_step = 32         # Pas de variation pour le nombre de neurones (ex: 32, 64, 96, 128)

dropout_min  = 0.2        # Taux de dropout minimal (régularisation)
dropout_max  = 0.2        # Taux de dropout maximal
dropout_step = 0.1        # Pas de variation pour le taux de dropout

activation_hidden = 'relu'      # Fonction d'activation pour les couches cachées (ReLU est courante)
activation_output = 'sigmoid'   # Fonction d'activation pour la couche de sortie (Sigmoid pour binaire)

optimizer            = 'adam'     # Algorithme d'optimisation (Adam est un choix populaire)
# learning_rate_values = [1e-3, 5e-4, 1e-4, 5e-5, 1e-5] # Exemple de valeurs pour le taux d'apprentissage si on l'optimisait
loss                 = 'binary_crossentropy' # Fonction de perte pour la classification binaire
metrics              = ['accuracy'] # Métriques à évaluer pendant l'entraînement (précision)

n_epochs_max = 200            # Nombre maximal d'époques pour l'entraînement
batch_size   = 300            # Taille des lots (batch) pour l'entraînement
patience     = 10             # Nombre d'époques sans amélioration avant d'arrêter l'entraînement (EarlyStopping)
verbose      = 1              # Niveau de verbosité pour l'affichage de l'entraînement

# Fonction qui génère une architecture de réseau dans un cadre donné

def build_model(hp):
    # Initialise un modèle Keras Sequential, un empilement linéaire de couches
    model = models.Sequential()

    # KerasTuner va faire varier le nombre de couches cachées entre n_layers_min et n_layers_max
    for i in range(hp.Int('num_layers', n_layers_min, n_layers_max)):
        # Ajoute une couche Dense (entièrement connectée)
        model.add(layers.Dense(units = hp.Int('units_' + str(i+1), # KerasTuner sélectionne le nombre de neurones pour cette couche
                                              min_value = n_units_min, # Minimum pour le nombre de neurones
                                              max_value = n_units_max, # Maximum pour le nombre de neurones
                                              step      = n_units_step), # Pas de variation
                               activation = activation_hidden, # Utilise la fonction d'activation définie pour les couches cachées
                               name = 'Hidden_layer_' + str(i+1))) # Nomme la couche pour faciliter l'identification
        # Ajoute une couche Dropout après chaque couche Dense pour la régularisation
        model.add(layers.Dropout(rate=hp.Float('dropout_' + str(i+1), # KerasTuner sélectionne le taux de dropout
                                                min_value = dropout_min, # Minimum pour le taux de dropout
                                                max_value = dropout_max, # Maximum pour le taux de dropout
                                                step      = dropout_step))) # Pas de variation

    # Ajoute la couche de sortie
    model.add(layers.Dense(dim_outputs, activation=activation_output, name='Output_layer'))

    # Compile le modèle : configure l'optimiseur, la fonction de perte et les métriques
    model.compile(optimizer = optimizer, # Utilise l'optimiseur défini ('adam')
                  # optimizer = keras.optimizers.Adam(learning_rate=learning_rate), # Exemple si le learning rate était tuné
                  loss      = loss,      # Utilise la fonction de perte définie ('binary_crossentropy')
                  metrics   = metrics)   # Utilise les métriques définies (['accuracy'])

    # Retourne le modèle Keras construit
    return model

### 2.2. Entraînement

In [18]:
# Paramètres de configuration du KerasTuner
objective              = 'val_loss'            # L'objectif à minimiser pour trouver le meilleur modèle (ici, la perte de validation)
n_trials               = 50                    # Nombre total de configurations d'hyperparamètres à tester
n_executions_per_trial = 1                     # Nombre de fois que chaque configuration est entraînée (pour réduire la variance)
directory              = 'Project'             # Répertoire où KerasTuner va sauvegarder les résultats de ses essais
project_name           = 'Spam_Tuner'          # Nom du projet KerasTuner

# Paramètres pour le callback EarlyStopping
monitor  = 'val_loss'                          # La métrique à surveiller pour l'arrêt anticipé
mode     = 'min'                               # Le mode de surveillance (minimiser la perte de validation)
patience = 10                                  # Nombre d'époques sans amélioration avant d'arrêter l'entraînement

# Paramètres d'entraînement généraux
batch_size   = 300                             # Taille des lots pour l'entraînement
n_epochs_max = 200                             # Nombre maximal d'époques pour l'entraînement de chaque modèle
verbose      = 0                               # Niveau de verbosité (0 = silencieux, 1 = barre de progression, 2 = une ligne par époque)

# Initialisation du tuner (RandomSearch recherche des hyperparamètres aléatoirement)
# génération de n_trials (ici 50 - très peu !!) réseaux avec hyperparamètres choisis aléatoirement

# définition de la fonction tuner
tuner = RandomSearch(build_model,                                  # La fonction qui construit et compile le modèle Keras
                     objective            = objective,             # L'objectif d'optimisation défini ci-dessus
                     max_trials           = n_trials,              # Le nombre maximum d'essais à exécuter
                     executions_per_trial = n_executions_per_trial,# Le nombre d'exécutions par essai
                     directory            = directory,             # Le répertoire de sauvegarde
                     project_name         = project_name)          # Le nom du projet

# Configuration du callback EarlyStopping
# Ce callback arrête l'entraînement si la métrique surveillée ne s'améliore pas après un certain nombre d'époques
callback = callbacks.EarlyStopping(monitor              = monitor,              # Moniteur la perte de validation
                                   mode                 = mode,                 # Cherche à minimiser la perte
                                   patience             = patience,             # Arrête après 10 époques sans amélioration
                                   restore_best_weights = True)                 # Restaure les poids du meilleur modèle

# Lancement de la fonction tuner : recherche d'hyperparamètres
tuner.search(X_train_norm, # Données d'entraînement normalisées
             y_train,      # Labels d'entraînement
             batch_size      = batch_size,          # Taille du lot
             epochs          = n_epochs_max,        # Nombre maximal d'époques
             validation_data = (X_valid_norm, y_valid), # Données de validation pour évaluer les performances
             callbacks       = [callback],          # Utilise le callback EarlyStopping
             verbose         = verbose)             # Niveau de verbosité

In [19]:
# donne les meilleurs hyperparamètres

best_hp = tuner.get_best_hyperparameters()[0]

print(f"Nombre de couches : {best_hp.get('num_layers')}")

for i in range(1, best_hp.get('num_layers') + 1):
    print(f"Nombre de neurones couche {i} : {best_hp.get(f'units_{i}')}")

Nombre de couches : 2
Nombre de neurones couche 1 : 32
Nombre de neurones couche 2 : 64


In [20]:
# récupère le meilleur modèle
best_model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### 2.3. Prévisions

In [21]:
y_test_pred = best_model.predict(X_test_norm)
y_test_pred[0:5]

29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


array([[9.9977511e-01],
       [3.3953849e-02],
       [3.6034505e-06],
       [8.6642420e-03],
       [7.9904530e-05]], dtype=float32)

In [22]:
y_test_pred_classes = (y_test_pred > 0.5).astype(int)
y_test_pred_classes[0:5]

array([[1],
       [0],
       [0],
       [0],
       [0]])

In [23]:
score_test = best_model.evaluate(X_test_norm, y_test, verbose=0)
print(f'Entropie test   : {score_test[0]:4.4f}')
print(f'Exactitude test : {score_test[1]:4.4f}')

Entropie test   : 0.1499
Exactitude test : 0.9490
